# Детекция ботов

Идея такая: по событиям внутри окна собираю признаки на одну cookie,
потом обучаю несколько моделей из sklearn и усредняю их ответы.

Признаков получилось больше, чем в бейзлайне,
но они все обычные: темп действий, типы событий, platform, user-agent, категории,
локации, поиск и координаты курсора.


## Как я строю признаки

События сначала фильтруются по индивидуальному окну cookie. После этого одна cookie
превращается в одну строку признаков. Я считаю не сырые id, а агрегаты: активность,
паузы между действиями, разнообразие объявлений и категорий, поиск, user-agent,
platform и координаты курсора.

In [ ]:
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier

from metric import precision_at_recall, recall_at_fpr


SEED = 42


# полный user-agent не беру как категорию, он слишком подробный
# вместо этого вытаскиваю только семейство браузера
def browser_from_ua(x):
    x = str(x).lower()
    if "yabrowser" in x:
        return "yandex"
    if "firefox" in x:
        return "firefox"
    if "chrome" in x:
        return "chrome"
    if "safari" in x:
        return "safari"
    return "other"


# По user-agent еще грубо определяю ОС
def os_from_ua(x):
    x = str(x).lower()
    if "android" in x:
        return "android"
    if "iphone" in x or "ipad" in x:
        return "ios"
    if "windows" in x:
        return "windows"
    if "mac os" in x or "macintosh" in x:
        return "mac"
    if "linux" in x:
        return "linux"
    return "other"


# Для долей бывает нулевой знаменатель, поэтому делю безопасно
def safe_div(a, b):
    return a / np.where(b == 0, 1, b)


# энтропия тут как простая мера разнообразия категорий и локаций
def entropy(s):
    p = s.value_counts(normalize=True)
    if len(p) == 0:
        return 0.0
    return float(-(p * np.log2(p)).sum())


def make_features(meta, events):
    meta_cols = ["cookie_id", "cookie_created_at", "window_start_ts", "window_end_ts"]

    # сначала приклеиваю к событиям границы окна и оставляю только нужный интервал
    # события после window_end_ts брать нельзя, это уже будущее
    ev = events.merge(meta[meta_cols], on="cookie_id", how="inner")
    ev = ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)].copy()

    # удаляю только полные дубли строк, разные события в одну секунду оставляю
    ev = ev.drop_duplicates()

    # platform привожу к одному виду, чтобы WEB/Web/web не были разными признаками
    ev["platform"] = ev.platform.astype(str).str.lower().replace({"desktop": "web", "iphone": "ios"})
    ev["browser"] = ev.user_agent.map(browser_from_ua)
    ev["os"] = ev.user_agent.map(os_from_ua)
    ev["hour"] = ev.event_ts.dt.hour
    ev["is_night"] = ev.hour.between(0, 5).astype(int)
    ev["has_pointer"] = ev.pointer_x.notna().astype(int)
    ev["has_query"] = ev.search_query.notna().astype(int)
    ev["query_len"] = ev.search_query.fillna("").astype(str).str.len()

    # технические слова в user-agent могут подсказать автоматические клиенты
    ua_low = ev.user_agent.astype(str).str.lower()
    for word in ["headless", "selenium", "python", "scrapy", "curl", "wget", "bot", "okhttp"]:
        ev[f"ua_{word}"] = ua_low.str.contains(word, regex=False).astype(int)

    # базовые агрегаты по активности, поиску, интересам и координатам
    g = ev.groupby("cookie_id")
    feat = g.agg(
        n_events=("event_ts", "size"),
        n_items=("item_id", "nunique"),
        n_categories=("item_category", "nunique"),
        n_locations=("item_location", "nunique"),
        n_queries=("search_query", "nunique"),
        n_event_types=("event_name", "nunique"),
        n_platforms=("platform", "nunique"),
        n_user_agents=("user_agent", "nunique"),
        search_page_mean=("search_page", "mean"),
        search_page_max=("search_page", "max"),
        search_page_std=("search_page", "std"),
        pointer_x_std=("pointer_x", "std"),
        pointer_y_std=("pointer_y", "std"),
        pointer_x_min=("pointer_x", "min"),
        pointer_x_max=("pointer_x", "max"),
        pointer_y_min=("pointer_y", "min"),
        pointer_y_max=("pointer_y", "max"),
        pointer_share=("has_pointer", "mean"),
        query_share=("has_query", "mean"),
        query_len_mean=("query_len", "mean"),
        n_hours=("hour", "nunique"),
        hour_mean=("hour", "mean"),
        night_share=("is_night", "mean"),
        cat_entropy=("item_category", entropy),
        loc_entropy=("item_location", entropy),
        ua_headless=("ua_headless", "max"),
        ua_selenium=("ua_selenium", "max"),
        ua_python=("ua_python", "max"),
        ua_scrapy=("ua_scrapy", "max"),
        ua_curl=("ua_curl", "max"),
        ua_wget=("ua_wget", "max"),
        ua_bot=("ua_bot", "max"),
        ua_okhttp=("ua_okhttp", "max"),
    ).reset_index()
    feat["pointer_x_range"] = feat.pointer_x_max - feat.pointer_x_min
    feat["pointer_y_range"] = feat.pointer_y_max - feat.pointer_y_min

    # временные признаки: у ботов часто бывают короткие или слишком ровные паузы
    ev = ev.sort_values(["cookie_id", "event_ts"])
    ev["gap"] = ev.groupby("cookie_id").event_ts.diff().dt.total_seconds()

    # сессии разделяю паузой больше 30 минут
    ev["new_session"] = ((ev.gap > 1800) | ev.gap.isna()).astype(int)

    time_feat = ev.groupby("cookie_id").agg(
        gap_mean=("gap", "mean"),
        gap_std=("gap", "std"),
        gap_min=("gap", "min"),
        gap_median=("gap", "median"),
        active_time=("event_ts", lambda x: (x.max() - x.min()).total_seconds()),
        n_sessions=("new_session", "sum"),
        fast_gap_1s=("gap", lambda x: (x.dropna() <= 1).mean()),
        fast_gap_5s=("gap", lambda x: (x.dropna() <= 5).mean()),
        fast_gap_10s=("gap", lambda x: (x.dropna() <= 10).mean()),
        long_gap_5m=("gap", lambda x: (x.dropna() >= 300).mean()),
        long_gap_30m=("gap", lambda x: (x.dropna() >= 1800).mean()),
    ).reset_index()
    feat = feat.merge(time_feat, on="cookie_id", how="left")

    # считаю, возвращалась ли cookie к одним и тем же объявлениям
    item_counts = ev.dropna(subset=["item_id"]).groupby(["cookie_id", "item_id"]).size().reset_index(name="cnt")
    item_feat = item_counts.groupby("cookie_id").agg(
        item_seen_max=("cnt", "max"),
        item_seen_mean=("cnt", "mean"),
        item_seen_once_share=("cnt", lambda x: (x == 1).mean()),
        item_seen_many_share=("cnt", lambda x: (x >= 3).mean()),
    ).reset_index()
    feat = feat.merge(item_feat, on="cookie_id", how="left")

    # координаты есть не у всех событий, но для web это полезный сигнал
    ptr = ev.dropna(subset=["pointer_x", "pointer_y"]).copy()
    if len(ptr) > 0:
        ptr = ptr.sort_values(["cookie_id", "event_ts"])
        ptr["dx"] = ptr.groupby("cookie_id").pointer_x.diff()
        ptr["dy"] = ptr.groupby("cookie_id").pointer_y.diff()
        ptr["dist"] = np.sqrt(ptr.dx ** 2 + ptr.dy ** 2)
        ptr_feat = ptr.groupby("cookie_id").agg(
            pointer_dist_mean=("dist", "mean"),
            pointer_dist_std=("dist", "std"),
            pointer_dist_max=("dist", "max"),
            pointer_same_share=("dist", lambda x: (x.fillna(0) < 1).mean()),
        ).reset_index()
        feat = feat.merge(ptr_feat, on="cookie_id", how="left")

    # для категориальных полей беру топ значений и считаю частоты по cookie
    for col in ["event_name", "platform", "browser", "os", "seller_type", "item_category", "item_location"]:
        top = ev[col].value_counts().head(20).index
        tmp = pd.crosstab(ev.cookie_id, ev[col])
        tmp = tmp.reindex(columns=top, fill_value=0)
        tmp.columns = [f"{col}_{c}" for c in tmp.columns]
        feat = feat.merge(tmp.reset_index(), on="cookie_id", how="left")

    # возвращаю все cookie из meta, если событий в окне не было, признаки станут нулями
    out = meta[["cookie_id", "cookie_created_at", "window_start_ts"]].merge(feat, on="cookie_id", how="left")
    out["cookie_age_days"] = (out.window_start_ts - out.cookie_created_at).dt.total_seconds() / 86400
    out["window_day"] = out.window_start_ts.dt.day
    out["created_hour"] = out.cookie_created_at.dt.hour
    out = out.drop(columns=["cookie_created_at", "window_start_ts"]).fillna(0)

    out["items_per_event"] = safe_div(out.n_items, out.n_events)
    out["queries_per_event"] = safe_div(out.n_queries, out.n_events)
    out["events_per_hour"] = safe_div(out.n_events, out.active_time / 3600 + 1)
    out["sessions_per_event"] = safe_div(out.n_sessions, out.n_events)

    prefixes = ("event_name_", "platform_", "browser_", "os_", "seller_type_", "item_category_", "item_location_")
    count_cols = [c for c in out.columns if c.startswith(prefixes)]
    for c in count_cols:
        out[c + "_share"] = safe_div(out[c], out.n_events)
    return out


#Три модели ранжируют немного по-разному, усреднение получилось устойчивее одной модели
def fit_models(X_train, y_train):
    models = [
        ExtraTreesClassifier(n_estimators=1000, min_samples_leaf=2, max_features="sqrt", class_weight="balanced", random_state=SEED, n_jobs=-1),
        HistGradientBoostingClassifier(max_iter=400, learning_rate=0.04, max_leaf_nodes=15, l2_regularization=0.1, random_state=SEED),
        RandomForestClassifier(n_estimators=600, min_samples_leaf=3, max_features="sqrt", class_weight="balanced_subsample", random_state=SEED + 1, n_jobs=-1),
    ]
    for model in models:
        model.fit(X_train, y_train)
    return models


# финальный score - взвешенное среднее, больший вес у HistGradientBoosting
# на временной проверке этот вариант лучше перенесся на поздние дни
def predict(models, X):
    return (
        0.35 * models[0].predict_proba(X)[:, 1]
        + 0.50 * models[1].predict_proba(X)[:, 1]
        + 0.15 * models[2].predict_proba(X)[:, 1]
    )

## Обучение и проверка

Сначала считаю простой baseline на двух признаках, потом основную модель.
Валидация по времени: test начинается позже train, поэтому проверяю модель на последних днях train.


## Почему такая валидация

Test начинается позже train, поэтому случайный split был бы слишком оптимистичным.
Я проверяю модель на последних днях train и сравниваю её с простым baseline.

In [2]:
date_cols = ["cookie_created_at", "window_start_ts", "window_end_ts"]
train = pd.read_csv("data/train.csv", parse_dates=date_cols)
test = pd.read_csv("data/test.csv", parse_dates=date_cols)
events = pd.read_csv("data/events.csv.gz", parse_dates=["event_ts"])

# собираю признаки сразу для train+test, чтобы набор one-hot колонок совпал
# target в make_features не передается, так что утечки здесь нет
all_meta = pd.concat([train.drop(columns="target"), test], ignore_index=True)
features_all = make_features(all_meta, events)
Xtr = train[["cookie_id"]].merge(features_all, on="cookie_id", how="left").fillna(0)
Xte = test[["cookie_id"]].merge(features_all, on="cookie_id", how="left").fillna(0)
cols = [c for c in Xtr.columns if c != "cookie_id"]
# выравниваю колонки test под train на всякий случай
Xte = Xte.reindex(columns=["cookie_id"] + cols, fill_value=0)
y = train.target.values

# baseline специально простой: только число событий и число разных объявлений
baseline_cols = ["n_events", "n_items"]
print("baseline:")
for day in ["2026-04-17", "2026-04-18", "2026-04-19"]:
    valid = train.window_start_ts.ge(day).values
    baseline = RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=3,
        random_state=SEED,
        n_jobs=-1,
    )
    baseline.fit(Xtr.loc[~valid, baseline_cols], y[~valid])
    p_base = baseline.predict_proba(Xtr.loc[valid, baseline_cols])[:, 1]
    print(day, round(precision_at_recall(y[valid], p_base), 4))

# основную модель проверяю тем же временным split-ом
print("main model:")
for day in ["2026-04-17", "2026-04-18", "2026-04-19"]:
    valid = train.window_start_ts.ge(day).values
    models = fit_models(Xtr.loc[~valid, cols], y[~valid])
    p = predict(models, Xtr.loc[valid, cols])
    print(day, len(cols), round(precision_at_recall(y[valid], p), 4), round(recall_at_fpr(y[valid], p, 0.01), 4))

# после проверки обучаю модели на всем train и считаю score для test
models = fit_models(Xtr[cols], y)
score = predict(models, Xte[cols])
sub = pd.DataFrame({"cookie_id": test.cookie_id, "score": score})
# порог не сохраняю, проверяющая система сама перебирает пороги для P@R0.7
sub.to_csv("submission.csv", index=False)
print("saved", Xtr.shape, Xte.shape, sub.score.min(), sub.score.max())


baseline:
2026-04-17 0.0931
2026-04-18 0.098
2026-04-19 0.0823
main model:
2026-04-17 181 0.7584 0.6125
2026-04-18 181 0.7333 0.5638
2026-04-19 181 0.8333 0.5952
saved (11091, 182) (4909, 182) 0.0016110025569059043 0.9887528230677026


## Что сделал

- оставил только события внутри окна наблюдения;
- убрал точные дубли;
- посчитал агрегаты по активности, времени, сессиям и быстрым паузам;
- добавил признаки по поиску, категориям, локациям, platform, браузеру и ОС;
- добавил простые признаки по координатам курсора;
- обучил три модели sklearn и усреднил ответы;
- сохранил `submission.csv` с колонками `cookie_id` и `score`.


## Вывод

Baseline на двух признаках почти не отделяет ботов. Основная модель лучше, потому что
использует не только количество событий, но и структуру поведения: темп действий,
поиск, категории, user-agent и координаты. Финальный файл `submission.csv` содержит
только `cookie_id` и `score`, порог отдельно подбирать не нужно.